# Runbook: запуск парсеров ClinicalTrials.gov и проверка данныхЭта тетрадь служит единым чек-листом для инженеров данных. В ней собрано всё,что необходимо для запуска парсеров ClinicalTrials.gov, прогонки тестов и быстройвизуальной проверки собранных данных перед выгрузкой в дашборд.

## 1. Подготовка окруженияРабота ведётся из корня репозитория `coop_theory_diploma`. Перед первым запускомрекомендуется создать виртуальное окружение и установить зависимости:```bashpython -m venv .venvsource .venv/bin/activatepip install -r requirements.txt```

In [ ]:
# Создание виртуального окружения и установка зависимостей (опционально)
# Уберите символы комментария, чтобы выполнить команды. Активацию окружения
# выполняйте в терминале перед запуском тетради.
# !python -m venv .venv
# !source .venv/bin/activate  # выполните в терминале, чтобы активировать окружение
# !pip install -r requirements.txt


In [ ]:
# Проверим активную версию Python и расположение интерпретатораimport sysprint(sys.version)print(sys.executable)

> 💡 Если зависимости уже установлены, повторный вызов `pip install` не требуется.
> Ячейка выше оставлена закомментированной для удобства.

## 2. Запуск автотестов для парсеровВ репозитории присутствуют тесты для нормализаторов и клиентских обёрток. Передсбором свежих данных стоит убедиться, что они проходят успешно.

In [ ]:
!pytest -q

## 3. Запуск высокоуровневых парсеровДалее можно перейти к сбору данных из ClinicalTrials.gov. Мы ограничиваем числозаписей в демонстрационных примерах, чтобы минимизировать нагрузку на API.При необходимости увеличьте `max_studies`.

In [ ]:
from data.parsers.clinical_trials import (    ClinicalTrialsBySponsorParser,    ClinicalTrialsExpressionParser,    DEFAULT_SPONSORS,)from data.parsers.clinical_trials_api import ClinicalTrialsClientimport pandas as pdclient = ClinicalTrialsClient()by_sponsor_parser = ClinicalTrialsBySponsorParser(    sponsors=list(DEFAULT_SPONSORS)[:3],  # для ускорения берём первые три компании    client=client,    max_studies=50,    batch_size=50,)by_sponsor_result = by_sponsor_parser.parse()by_sponsor_records = by_sponsor_result.payload["records"]pd.DataFrame(by_sponsor_records)

Помимо агрегирования по компаниям, можно запрашивать произвольные выражения,например для валидации отдельных направлений исследований.

In [ ]:
expression_parser = ClinicalTrialsExpressionParser(    expr="oncology AND Recruiting",    client=client,    max_studies=50,    batch_size=50,)expression_record = expression_parser.parse().payload["records"][0]expression_record

## 4. Быстрая визуальная проверка агрегатовПосле получения данных удобно посмотреть на распределения фаз и статусов, чтобыубедиться, что парсеры вернули ожидаемые значения.

In [ ]:
phase_counts = (    pd.DataFrame(by_sponsor_records)    .set_index("name")["phase_counts"]    .apply(pd.Series)    .fillna(0)    .astype(int)    .sort_index(axis=1))status_counts = (    pd.DataFrame(by_sponsor_records)    .set_index("name")["status_counts"]    .apply(pd.Series)    .fillna(0)    .astype(int)    .sort_index(axis=1))phase_counts

In [ ]:
import plotly.express as pxphase_long = phase_counts.reset_index().melt(    id_vars="name", value_name="count", var_name="phase")fig_phase = px.bar(    phase_long,    x="name",    y="count",    color="phase",    title="Распределение исследований по фазам",)fig_phase.show()status_long = status_counts.reset_index().melt(    id_vars="name", value_name="count", var_name="status")fig_status = px.bar(    status_long,    x="name",    y="count",    color="status",    title="Распределение исследований по статусам",)fig_status.show()

## 5. Сохранение результата в JSONДля воспроизводимой выгрузки рекомендуется использовать готовый CLI-скрипт.Пример ниже сохраняет небольшой набор данных в `data/clinical_trials_sample.json`.

In [ ]:
!python data_fetch/build_clinical_trials_dataset.py \    --sponsor Pfizer \    --sponsor BIOCAD \    --max-studies 100 \    --batch-size 50 \    --out data/clinical_trials_sample.json

In [ ]:
from pathlib import Pathimport jsonoutput_path = Path("data/clinical_trials_sample.json")with output_path.open("r", encoding="utf-8") as fh:    dataset = json.load(fh)dataset.keys()

In [ ]:
pd.DataFrame(dataset.get("sponsors", []))

## 6. Следующие шаги* При необходимости расширьте список компаний или задайте собственное выражение  для `ClinicalTrialsExpressionParser`.* Результаты можно напрямую использовать в дашборде (`app/data.py`) либо  дополнительно обработать в аналитических тетрадях.* Не забывайте обновлять зависимости и периодически пересматривать тесты при  доработках нормализации.